# Python 中是如何进行内存管理的

以 **CPython** 为主来讲 Python 的内存管理。严格来说，不同 Python 实现的细节会有差异，但下面这些机制是最常见、也最值得掌握的：

- **对象和引用**：变量只是对象的引用，并不直接“装着值”。
- **引用计数**：大多数对象的生命周期由引用数量决定。
- **垃圾回收**：专门处理循环引用等引用计数无法解决的问题。
- **内存分配器**：CPython 还会自己管理小对象的内存分配与复用。

一句话总览：**CPython 的内存管理 = 引用计数（主力）+ 分代垃圾回收（兜底循环引用）+ 内存池 pymalloc（性能优化）**。

理解这几层之后，就能更清楚地看懂 `id()`、`del`、`gc.collect()`、循环引用、对象复用等现象。


## 一、变量不是对象本身，而是对象的引用

在 Python 里，赋值语句的本质是“把名字绑定到对象上”。

例如：

```python
a = [1, 2, 3]
b = a
```

此时 `a` 和 `b` 指向同一个列表对象。修改其中一个引用指向的对象内容，另一个名字也会看到变化。

In [ ]:
a = [1, 2, 3]
b = a

print(id(a))
print(id(b))
print(a is b)

a.append(4)
print(a)
print(b)

这里 `a is b` 为 `True`，说明两个名字绑定的是同一个对象。

这也解释了为什么 Python 的函数传参常被称为“按对象引用传递”或“按对象共享传递”：函数得到的是对象引用的副本，而不是对象本身的拷贝。

## 二、引用计数：对象何时被释放的第一道机制

CPython 最核心的内存管理策略是 **引用计数**。

简单理解：

- 每个对象都会记录“当前有多少个引用指向自己”。
- 当引用数变成 0 时，对象通常会立刻被销毁。
- 对象销毁时，和它关联的资源也会随之释放。

这也是为什么在很多场景里，Python 对象看起来“很及时”地被释放了。

In [1]:
import sys

x = []
print(sys.getrefcount(x))  # 这里会比真实值多 1，因为函数参数也会临时产生一次引用

y = x
print(sys.getrefcount(x))

del y
print(sys.getrefcount(x))

2
3
2


### `del` 的真实作用

`del x` 不是“删除对象”，而是**删除名字和对象之间的绑定关系**。

如果这个对象还有其他引用，它就还会继续存在；只有当最后一个引用也消失时，它才会真正进入释放流程。

In [ ]:
a = [1, 2, 3]
b = a
del a

print(b)  # 对象仍然存在，因为 b 还在引用它
print(id(b))

## 三、强引用与弱引用

前面说的引用，默认都是 **强引用（strong reference）**：

- 强引用会把对象的引用计数 +1，只要还有强引用存在，对象就绝不会被回收。
- 我们平时写的赋值、容器持有、传参，创建的全是强引用。

与之相对的是 **弱引用（weak reference）**，由标准库 `weakref` 提供：

- 弱引用 **不增加引用计数**，只是“顺手记录”某个对象，不影响它的生死。
- 一旦对象的强引用全部消失，对象照常回收，弱引用随之失效——之后访问它只能得到 `None`。

弱引用最典型的用途是 **缓存**：缓存不应该阻止对象被回收，否则缓存本身就成了变相的内存泄漏。


In [ ]:
import sys
import weakref


class Node:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return f"Node({self.name!r})"


obj = Node("A")
r = weakref.ref(obj)          # 创建弱引用，引用计数不变

print(sys.getrefcount(obj))   # 只有 obj 本身 + 传参的临时引用，r 没有贡献计数
print(r())                    # 对象还活着，r() 返回对象本身

del obj                       # 删掉唯一的强引用
print(r())                    # None：对象已被回收，弱引用自动失效


两点注意：

- 并非所有对象都支持弱引用：对 `list`、`dict`、`int`、`str` 等内置类型直接 `weakref.ref(...)` 会抛 `TypeError`；自定义类的实例可以。
- `weakref` 还提供“弱引用版容器”，最常用的是 **`WeakValueDictionary`**：值以弱引用保存，别处没有强引用时条目自动消失，适合做缓存。


In [ ]:
import weakref


class Expensive:
    def __init__(self, name):
        self.name = name


cache = weakref.WeakValueDictionary()

tmp = Expensive("a")
cache["a"] = tmp              # 值以弱引用方式保存
print("有强引用时:", list(cache.keys()))

del tmp                       # 唯一的强引用消失
print("强引用消失后:", list(cache.keys()))   # 条目自动消失，对象被回收


## 四、为什么还需要垃圾回收：循环引用

引用计数很高效，但它有一个天然问题：**循环引用**。

例如两个对象互相引用：

- A 引用 B
- B 又引用 A

即使外部已经没有任何名字指向它们，它们彼此之间仍然“撑着”对方的引用计数，导致无法自动释放。

所以 CPython 还提供了 **垃圾回收器（GC）**，专门处理这类引用计数难以独立解决的问题。

In [2]:
import gc
import  sys

class Node:
    def __init__(self, name):
        self.name = name
        self.other = None


a = Node('A')
b = Node('B')
a.other = b
b.other = a

print(f"对象a的计数器:{sys.getrefcount(a)}")
print(f"对象b的计数器:{sys.getrefcount(b)}")

print(gc.isenabled())
print(gc.get_count())

del a
del b

# 这时如果没有 GC，循环引用对象可能无法被及时回收
print(gc.collect())

对象a的计数器:3
对象b的计数器:3
True
(238, 7, 0)
5


### 分代回收

CPython 的 GC 采用 **分代回收** 思想，理论基础是“弱代假设”：大多数对象都是朝生夕死的，新对象更可能很快变成垃圾。

- 对象分为 **0、1、2 三代**，新创建的容器对象先进入 0 代。
- 每熬过一轮扫描仍存活，就晋升到下一代；代数越高，扫描频率越低。
- 触发时机由阈值控制：`gc.get_threshold()` 默认返回 `(2000, 10, 10)`（Python 3.12+；更早版本为 `(700, 10, 10)`），大致含义是——0 代每净增约 700 个容器对象就扫一次；0 代扫 10 次才触发一次 1 代扫描；1 代扫 10 次才触发一次 2 代全量扫描。
- 常用接口：`gc.collect()` 手动回收、`gc.disable()` / `gc.enable()` 开关。

要注意：这三代 **不是三块内存区域**，只是三条“待追踪容器对象”的链表，决定的是扫描频率。这一点在后面与 JVM 新生代/老年代对比时还会再强调。


## 五、内存池 pymalloc：小对象为什么快

CPython 为了提高性能，不会每次创建对象都直接找操作系统要内存，而是有自己的分配策略——**内存池（pymalloc）**。

频繁地向系统 `malloc`/`free` 有两个问题：

- 系统调用和 `malloc` 本身的开销，对“几十字节的小对象”来说太贵了；
- 反复申请、释放不同大小的内存，会产生 **内存碎片**。

pymalloc 的策略：

1. **只服务 ≤ 512 字节的小对象**；更大的对象（如很长的 `bytes`、大 `list` 的缓冲区）直接走系统 `malloc`。
2. 内存按三级组织：

   ```text
   arena（256 KB，向操作系统申请的大块）
     └── pool（4 KB）
           └── block（实际分给对象的“格子”，按 8 字节对齐分成固定档位）
   ```

3. 同一个 pool 里的 block 都是同一档位；释放时 block 挂回该档位的空闲链表 **复用**，而不是还给操作系统。
4. 只有整个 arena 完全空闲时，CPython 才可能把这块内存归还给操作系统。

这解释了一个经典现象：**`del` 掉对象之后，进程占用的内存（RSS）往往不会立刻下降**——对象确实释放了，但内存只是回到池子里等着复用；要真正还给 OS，得等整个 arena 空闲，实践中常常凑不齐。这是特性，不是泄漏。


In [ ]:
# 演示 1：block 被回收后立刻复用
x = [1, 2, 3]
addr = id(x)
print(f"x 的地址: {addr}")

del x                  # 列表销毁，占用的 block 回到池子
y = [4, 5, 6]
print(f"y 的地址: {id(y)}")
print("复用了同一块内存:", id(y) == addr)


In [ ]:
# 演示 2：Python 层面已经释放，不代表进程内存一定还给操作系统
import tracemalloc

tracemalloc.start()

data = [bytes(64) for _ in range(1000)]
current, peak = tracemalloc.get_traced_memory()
print(f"持有 1000 个小对象: Python 层占用约 {current / 1024:.0f} KB")

data.clear()
current, _ = tracemalloc.get_traced_memory()
print(f"清空列表之后:      Python 层占用约 {current / 1024:.0f} KB（基本归零）")
# 但进程的 RSS 未必同步下降——内存大概率还留在分配器的池子里等着复用


## 六、CPython 的内存区域，以及与 JVM 内存模型的对比

熟悉 JVM 的同学常会问：Python 有没有自己的“堆、栈、元空间、新生代、老年代”？

先看 CPython 进程的内存里实际有什么：

| CPython 中的区域 | 存什么 | 类比 JVM |
| --- | --- | --- |
| 静态区 / 全局区 | 解释器自身的数据、小整数缓存、驻留字符串、`None`/`True`/`False` 单例 | 常量池、静态变量 |
| 对象堆 | **所有** Python 对象：pymalloc 池化的小对象 + 走 `malloc` 的大对象 | 堆 |
| 帧数据区 | 函数调用帧（局部变量就在帧里，帧本身也是解释器托管的对象） | 虚拟机栈（线程私有） |
| C 栈 | 解释器执行时的 C 层递归，`RecursionError` 与它有关 | 本地方法栈 |
| （没有独立区域） | 类、函数、字节码都是普通堆对象，同样有引用计数 | 方法区 / 元空间 |


对照 JVM 的经典内存模型，有三个关键差异：

### 1. 栈：Python 没有 JVM 式的线程栈

JVM 能把局部变量放在线程栈上，前提是它有原始类型（`int` 等无需装箱的值）。CPython 里 **一切皆 `PyObject*`**，连整数都是堆上的完整对象，变量只是指向它的指针，所以“局部变量表”只能放进函数调用帧——帧本身也是对象（因此才有 `inspect`、闭包、生成器挂起这些能力）。

### 2. 分代：名字一样，实质不同

|  | JVM 新生代 / 老年代 | CPython 0 / 1 / 2 代 |
| --- | --- | --- |
| 覆盖范围 | 整个堆，所有对象按年龄搬移 | 只登记被追踪的容器对象，`int`/`str` 根本不进分代体系 |
| 形态 | 真实的堆区域 | 三条链表；晋升只是换链表，**内存位置不动** |
| 配套算法 | 复制、标记-整理等，回收时压缩搬移 | 只决定扫描频率，回收主力仍是引用计数 |

CPython 的对象 **永远不会被搬移**：`id()` 就是对象的内存地址，C 扩展手里还握着裸指针，一旦搬移全部失效。这等于排除了压缩式 GC——碎片只能靠 pymalloc 按 8 字节分档的池化来缓解。

### 3. “不同区用不同算法” → “不同对象用不同机制”

JVM 按空间分工：新生代复制算法、老年代标记-整理、元空间独立分配。CPython 按 **对象类别** 分工：

```text
所有对象      → 引用计数（归零即回收，确定性）
容器对象      → 环收集器（只补循环引用的窟窿）
≤512B 小对象 → pymalloc 池化分配
大对象        → 系统 malloc
```

一句话总结：**JVM 是“回收器中心”设计——一切交给 GC，所以要按区域分代、配不同算法；CPython 是“引用计数中心”设计——回收主力确定性兜底，GC 只做环检测补丁，因此不需要按年龄分区搬移，也没有给元数据单开区域的必要：类和代码在 Python 里只是普通对象，和别的对象共用同一套堆、同一套计数、同一个 GC。**


## 七、一些常见“内存现象”

### 1. 小整数缓存
CPython 会对部分小整数做缓存，因此你可能会看到一些整数对象的 `id()` 相同或表现出复用现象。

### 2. 字符串驻留
一些字符串字面量也可能被复用，以减少重复分配。

### 3. `is` 和 `==` 不要混淆
- `is` 比较的是是不是同一个对象。
- `==` 比较的是值是否相等。

很多“内存管理”的疑问，最后都会落到这三个点上。

In [ ]:
a = 256
b = 256
print(a is b)

x = 1000
y = 1000
print(x == y)
print(x is y)

s1 = 'hello'
s2 = 'hello'
print(s1 is s2)
print(s1 == s2)

## 八、实战建议

写 Python 程序时，和内存管理相关的实用建议主要有：

- 尽量避免不必要的对象持有，尤其是大对象和长生命周期缓存；缓存要有上限（`functools.lru_cache(maxsize=...)`）或用弱引用（`WeakValueDictionary`）。
- 小心循环引用，特别是对象之间互相持有时；确实需要“反向指回去”的场景，考虑 `weakref`。
- 大量小实例可以用 `__slots__` 省掉每个实例的 `__dict__`，内存常能省下一半。
- 不要把 `del` 理解成“立刻把内存还给系统”：对象释放了，内存也可能先留在池子里复用。
- 排查内存问题时，优先看对象引用关系、缓存、容器持有情况。
- 需要深入分析时，配合 `gc`（`gc.collect()`、`gc.get_referrers()`）、`sys.getrefcount()`、`tracemalloc` 等工具。

一句话总结：**Python 内存管理的核心是引用计数，垃圾回收负责补足循环引用，分配器负责性能和复用。**


## 九、面试八股文：高频面试题与参考答案

按“总览 → GC → 内存池 → 实战 → 看代码题”组织。建议先遮住答案自己答一遍，再对照参考答案；“常见追问”是面试官继续深挖的方向。


### 第一组：总览与引用计数

**Q1. 简述 Python 的内存管理机制（必考开场题）**

参考答案（三句话）：

- **引用计数为主**：每个对象头部有 `ob_refcnt`，归零立即销毁；
- **分代 GC 为辅**：专门回收引用计数解决不了的循环引用；
- **pymalloc 内存池**：≤ 512 字节小对象走 arena/pool/block 池化分配，大对象走系统 `malloc`。

前提：变量只是对象的引用，所有对象都在堆上。

> 常见追问：为什么 Python 选引用计数而不是 Java 那种追踪式 GC？—— 回收确定、及时（RAII 式），实现简单；代价是循环引用问题和计数维护开销。

**Q2. 引用计数什么时候 +1、-1？优缺点？**

- **+1**：赋值给新名字、存入容器、作为参数传入。
- **-1**：`del` 名字、离开作用域、从容器移除、函数返回。
- 优点：实时、确定性强。缺点：循环引用无法回收；每次赋值都要维护计数，有开销。

> 常见追问①：`sys.getrefcount` 为什么总比真实值多 1？—— 传参本身产生一次临时引用。
> 常见追问②：Python 3.12+ 对 `None` 查计数为什么返回巨大值？—— 不朽对象（PEP 683），计数被固定，永不销毁。

**Q3. `del x` 会立即释放内存吗？**

- `del` 只删除名字与对象的绑定，引用计数 -1；对象是否销毁只取决于还有没有其他引用。
- 即使对象销毁，内存也只是回到 pymalloc 池子里复用，不一定还给操作系统——`del` ≠ “释放内存到系统”。

**Q4. `is` 和 `==` 的区别？`a = 256; b = 256` 时 `a is b` 呢？**

- `is` 比较对象身份（`id()`），`==` 比较值。
- -5～256 的小整数是预创建的单例，`is` 为 `True`；字符串字面量会驻留复用。
- 结论：**判等一律用 `==`，`is` 只用于 `x is None` 这类身份判断**。


### 第二组：垃圾回收

**Q5. 什么是循环引用？CPython 怎么解决？**

- A 引用 B、B 引用 A，外部已不可达，但互相撑着计数，永不为 0。
- CPython 用**环收集器**：只追踪“容器对象”（list/dict/类实例等），从根对象做可达性分析，找出“外部不可达、只在内部成环”的孤立区，整体回收。`int`/`str` 不可能成环，根本不进这个体系。

> 常见追问①：带 `__del__` 的循环引用能回收吗？—— Python 3.4+（PEP 442）之后能；更早版本会进 `gc.garbage` 成为不可回收垃圾。
> 常见追问②：`gc.collect()` 的返回值是什么？—— 本次回收到的不可达对象数。

**Q6. 讲讲分代回收，和 JVM 的新生代/老年代一样吗？（高分题）**

- 0/1/2 三代，基于“弱代假设”（新对象大多早死）：新容器对象进 0 代，熬过一轮扫描晋升，代越高扫描越少；阈值 `gc.get_threshold()` 默认 `(2000, 10, 10)`（3.12+；旧版本为 `(700, 10, 10)`）。
- 关键差异（答出这条就超过多数人）：**CPython 的“代”不是内存分区，只是三条待扫描链表；只覆盖容器对象；对象晋升不搬移内存；回收主力仍然是引用计数**。JVM 的分代是真实堆布局，配复制/整理算法，对象会搬移。

**Q7. 为什么 CPython 不用标记-整理（压缩）算法？**

- 因为对象**永远不能搬移**：`id()` 在 CPython 里就是内存地址，C 扩展手里握着裸 `PyObject*` 指针，搬移等于全部失效。
- 所以碎片只能靠 pymalloc 按尺寸分档来缓解，而不是压缩。


### 第三组：内存池

**Q8. 讲讲 pymalloc？为什么 `del` 掉大对象后进程内存不降？**

- 只服务 **≤ 512 字节** 的小对象，三级结构：arena（256 KB，向 OS 申请）→ pool（4 KB）→ block（8 字节对齐的固定档位）。
- 释放的 block 挂回空闲链表**复用**而不是还 OS，只有整个 arena 空闲才可能归还；大对象直接走系统 `malloc`。
- 所以 RSS 不降是特性不是泄漏——内存留在池子里等下一个对象用。


### 第四组：实战

**Q9. Python 有内存泄漏吗？什么场景？**

有。引用计数 + GC 只保证“**不可达**对象”被回收，“**可达但没用**”的对象照样泄漏：

- 无上限的全局缓存/容器；
- 闭包、回调注册、日志 handler 意外持有大对象；
- 循环引用抱着大对象长期不释放；
- C 扩展自身泄漏。

防范：缓存设上限（`functools.lru_cache(maxsize=...)`）或用弱引用。

**Q10. 强引用和弱引用？`weakref` 什么场景用？**

- 强引用计数 +1，撑住对象；弱引用不增加计数，对象死后访问得 `None`。
- 场景：`WeakValueDictionary` / `WeakSet` 做不阻止回收的缓存、观察者列表、父子节点反向引用。
- 注意：内置 `list` / `dict` 不支持弱引用。

> 常见追问：`__slots__` 为什么省内存？—— 省掉每个实例的 `__dict__`，属性变成类级固定槽位，大量小实例能省一半；副作用是不能动态加属性，且需要 `__weakref__` 槽才支持弱引用。

**Q11. 线上怎么排查内存问题？**

- 标准库：`tracemalloc`（拍两个快照对比增长点）、`gc.get_referrers(obj)`（查谁引用着它）、`gc.get_objects()`。
- 第三方：`objgraph`（引用链可视化）、`memory_profiler`。
- 注意区分 Python 层内存和进程 RSS。


### 第五组：看代码题（先自己猜输出，再运行验证）

**C1. 引用计数**


In [ ]:
import sys

a = [1, 2, 3]
b = a
print(sys.getrefcount(a))   # 先猜输出
del b
print(sys.getrefcount(a))   # 再猜输出


**答案：`3` 和 `2`** —— `a`、`b` 各持一个强引用，`getrefcount` 的传参再临时 +1，所以是 3；`del b` 后剩 `a` + 传参，是 2。

**C2. 小整数缓存（有坑，区分度最高）**


In [ ]:
a = 256
b = 256
c = 257
d = 257
print(a is b, c is d)       # 在 REPL 逐行执行、和整体运行，结果一样吗？


**答案：Jupyter 单元格 / `.py` 文件里整体编译，输出 `True True`；REPL 逐行执行输出 `True False`。** 小整数缓存只覆盖 -5～256，所以逐行跑时 257 是两个对象；但同一个代码块里的常量会被编译器合并成一个，整体运行时 257 也相同。考点：`is` 的结果依赖实现细节，判等不能依赖 `is`。

**C3. 循环引用与 `gc.disable()`**


In [ ]:
import gc


class Node:
    pass


a, b = Node(), Node()
a.other = b
b.other = a

del a, b
gc.disable()
# 此时这对循环对象还会被回收吗？手动 gc.collect() 呢？


**答案：`del` 之后这两个对象已不可达，且早已进入 GC 的追踪体系——GC 自动触发开着时，下次 0 代扫描就会收走它们；`gc.disable()` 关闭的只是自动触发，对象仍留在追踪链表里，手动 `gc.collect()` 依然能回收。所以 disable ≠ 泄漏，但循环垃圾会一直攒到被回收为止。**
